# Mis Soluciones: Limpieza y Preprocesamiento de Datos

<div style="background: linear-gradient(135deg, #2c3e50 0%, #3498db 100%); padding: 30px; border-radius: 10px; color: white; margin-bottom: 20px;">
<h2 style="color: white; margin-top: 0;">30 ejercicios practicos para dominar la limpieza de datos</h2>
<p style="font-size: 16px;">Cada ejercicio incluye enunciado, resolucion detallada y codigo con visualizacion antes/despues.</p>
</div>

---

## Indice de Ejercicios

| # | Bloque | Ejercicio |
|---|--------|-----------|
| **Bloque 1** | **Exploracion** | |
| 1 | Exploracion | Perfilar df_dirty: shape, dtypes, nulls, describe |
| 2 | Exploracion | Identificar columnas con >5% nulls, tipos incorrectos, constantes |
| 3 | Exploracion | Generar reporte de calidad (tabla con % nulls, tipo, n_unique por columna) |
| 4 | Exploracion | Heatmap de missing values |
| **Bloque 2** | **Valores Nulos** | |
| 5 | Valores Nulos | Imputar 'edad' con media vs mediana -- comparar histogramas |
| 6 | Valores Nulos | Imputar 'departamento' (categorica) con moda |
| 7 | Valores Nulos | Serie temporal simulada, imputar con forward-fill |
| 8 | Valores Nulos | Imputar 'salario' por grupo (media por departamento) |
| 9 | Valores Nulos | KNN Imputer en columnas numericas -- comparar con media |
| 10 | Valores Nulos | Crear columna 'salario_missing' como flag antes de imputar |
| **Bloque 3** | **Duplicados** | |
| 11 | Duplicados | Detectar y eliminar duplicados exactos |
| 12 | Duplicados | Detectar duplicados aproximados en 'nombre' |
| 13 | Duplicados | Duplicados por subconjunto ['nombre', 'departamento'] |
| **Bloque 4** | **Outliers** | |
| 14 | Outliers | Detectar outliers en 'salario' con IQR, boxplot |
| 15 | Outliers | Detectar outliers en 'edad' con Z-score |
| 16 | Outliers | Winsorizar 'salario' al percentil 1-99 vs eliminar |
| 17 | Outliers | Isolation Forest en ['edad', 'salario', 'evaluacion'] |
| **Bloque 5** | **Tipos y Validacion** | |
| 18 | Tipos y Validacion | Limpiar columna "$1,234.56" y convertir a float |
| 19 | Tipos y Validacion | Parsear 'fecha_ingreso' con formatos mixtos |
| 20 | Tipos y Validacion | Validar rangos: edad [18,70], salario > 0, evaluacion [1,10] |
| **Bloque 6** | **Limpieza de Strings** | |
| 21 | Limpieza de Strings | Normalizar 'nombre': strip, title case, espacios multiples |
| 22 | Limpieza de Strings | Extraer dominio de emails con regex |
| 23 | Limpieza de Strings | Estandarizar 'departamento': title case, corregir inconsistencias |
| **Bloque 7** | **Encoding** | |
| 24 | Encoding | One-Hot Encoding de 'ciudad' con drop_first |
| 25 | Encoding | Ordinal Encoding de 'satisfaccion' |
| 26 | Encoding | Target Encoding de 'departamento' con media de evaluacion |
| **Bloque 8** | **Escalado** | |
| 27 | Escalado | Comparar MinMax vs Standard vs Robust en 'salario' |
| 28 | Escalado | ColumnTransformer: escalar numericas + one-hot categoricas |
| **Bloque 9** | **Pipelines** | |
| 29 | Pipelines | Pipeline completo: imputar + escalar numericas, imputar + one-hot categoricas |
| 30 | Pipelines | Limpieza end-to-end: df_dirty a df_clean listo para modelo |

**Instrucciones:** Resuelve cada ejercicio en la celda de codigo vacia. Consulta el notebook de ejercicios original solo si te atascas.

In [ ]:
# === IMPORTS Y CONFIGURACION ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                   MaxAbsScaler, LabelEncoder, OneHotEncoder,
                                   OrdinalEncoder, PowerTransformer)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from scipy import stats
from IPython.display import display, HTML

# Estilos
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>.output_result { max-width:100% !important; }</style>"))
plt.style.use('seaborn-v0_8-whitegrid')

C_PRIMARY = '#3498db'
C_DANGER = '#e74c3c'
C_SUCCESS = '#2ecc71'
C_DARK = '#2c3e50'
C_ORANGE = '#f39c12'

# === CREAR DATASET SINTETICO "SUCIO" ===
np.random.seed(42)
n = 500

df_dirty = pd.DataFrame({
    'id': range(1, n + 1),
    'nombre': np.random.choice(
        ['Juan Garcia', 'MARIA LOPEZ', '  pedro martinez  ', 'Ana Torres',
         'LUIS sanchez', None, 'pedro Martinez'], n),
    'edad': np.where(np.random.random(n) < 0.1, np.nan,
                     np.random.normal(35, 12, n).astype(int)),
    'salario': np.where(np.random.random(n) < 0.08, np.nan,
                        np.random.lognormal(10.5, 0.5, n)),
    'departamento': np.random.choice(
        ['Ventas', 'ventas', 'VENTAS', 'IT', 'it', 'Marketing', 'RRHH', None], n),
    'fecha_ingreso': [
        f'2020-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}'
        if np.random.random() > 0.1
        else (f'{np.random.randint(1,29)}/{np.random.randint(1,13)}/2020'
              if np.random.random() > 0.5 else None)
        for _ in range(n)
    ],
    'satisfaccion': np.random.choice(
        ['bajo', 'medio', 'alto', 'Bajo', 'ALTO', None], n),
    'evaluacion': np.where(np.random.random(n) < 0.05, np.nan,
                           np.round(np.random.uniform(1, 10, n), 1)),
    'ciudad': np.random.choice(
        ['Madrid', 'Barcelona', 'Sevilla', 'Valencia', 'madrid', 'BARCELONA'], n),
})

# Inyectar outliers
df_dirty.loc[10, 'salario'] = 500000
df_dirty.loc[20, 'edad'] = 150
df_dirty.loc[30, 'edad'] = -5

# Inyectar duplicados (15 filas duplicadas)
df_dirty = pd.concat([df_dirty, df_dirty.iloc[:15]], ignore_index=True)

print(f"Dataset sucio creado: {df_dirty.shape[0]} filas x {df_dirty.shape[1]} columnas")
print(f"Problemas inyectados: nulos, duplicados, outliers, tipos incorrectos, strings inconsistentes, fechas mixtas")
df_dirty.head(10)

**Ejercicio 1:** Perfilar el DataFrame `df_dirty`. Obtener su forma (shape), tipos de datos (dtypes), conteo de valores nulos por columna y estadisticas descriptivas (describe). Interpretar los resultados para identificar los problemas mas evidentes del dataset.

In [ ]:
# Tu solucion para el Ejercicio 1
# Escribe tu codigo aqui



**Ejercicio 2:** Identificar las columnas que tienen mas del 5% de valores nulos, las columnas con tipos de datos incorrectos (por ejemplo, columnas que deberian ser numericas pero son `object`) y las columnas constantes (con un solo valor unico). Presentar los resultados de forma estructurada.

In [ ]:
# Tu solucion para el Ejercicio 2
# Escribe tu codigo aqui



**Ejercicio 3:** Generar un reporte de calidad de datos en forma de tabla que muestre, para cada columna: nombre, tipo de dato, numero de valores nulos, porcentaje de nulos, numero de valores unicos y un ejemplo de valor. Ordenar por porcentaje de nulos de mayor a menor.

In [ ]:
# Tu solucion para el Ejercicio 3
# Escribe tu codigo aqui



**Ejercicio 4:** Crear un heatmap de valores faltantes (missing values) del DataFrame `df_dirty`. Utilizar seaborn o matplotlib para visualizar la matriz de nulidad, donde cada fila es una observacion y cada columna es una variable, coloreando en un tono distinto las celdas con datos faltantes.

In [ ]:
# Tu solucion para el Ejercicio 4
# Escribe tu codigo aqui



**Ejercicio 5:** Imputar los valores nulos de la columna `edad` de dos formas: con la media y con la mediana. Comparar ambas estrategias visualmente mediante histogramas antes y despues de la imputacion. Discutir cual es mas apropiada cuando existen outliers.

In [ ]:
# Tu solucion para el Ejercicio 5
# Escribe tu codigo aqui



**Ejercicio 6:** Imputar los valores nulos de la columna `departamento` (categorica) utilizando la moda (valor mas frecuente). Mostrar la distribucion de valores antes y despues de la imputacion con un grafico de barras.

In [ ]:
# Tu solucion para el Ejercicio 6
# Escribe tu codigo aqui



**Ejercicio 7:** Crear una serie temporal simulada con valores faltantes e imputar utilizando forward-fill (propagacion hacia adelante). Visualizar la serie antes y despues de la imputacion para verificar que los huecos se rellenan con el ultimo valor valido.

In [ ]:
# Tu solucion para el Ejercicio 7
# Escribe tu codigo aqui



**Ejercicio 8:** Imputar los valores nulos de `salario` utilizando la media de salario por departamento (imputacion por grupo). Primero normalizar `departamento` a title case, luego calcular la media de salario para cada departamento y rellenar los nulos con la media correspondiente a su grupo.

In [ ]:
# Tu solucion para el Ejercicio 8
# Escribe tu codigo aqui



**Ejercicio 9:** Utilizar `KNNImputer` de scikit-learn para imputar las columnas numericas (`edad`, `salario`, `evaluacion`) y comparar el resultado con la imputacion por media simple. Visualizar las distribuciones resultantes de ambos metodos.

In [ ]:
# Tu solucion para el Ejercicio 9
# Escribe tu codigo aqui



**Ejercicio 10:** Antes de imputar `salario`, crear una columna binaria `salario_missing` que valga 1 donde `salario` es NaN y 0 donde tiene valor. Luego imputar `salario` con la mediana. Mostrar la tabla con ambas columnas y verificar que la flag se creo correctamente.

In [ ]:
# Tu solucion para el Ejercicio 10
# Escribe tu codigo aqui



**Ejercicio 11:** Detectar y eliminar los duplicados exactos en `df_dirty`. Mostrar cuantos duplicados hay, cuales son las filas duplicadas, y el shape del DataFrame antes y despues de la eliminacion.

In [ ]:
# Tu solucion para el Ejercicio 11
# Escribe tu codigo aqui



**Ejercicio 12:** Detectar duplicados aproximados en la columna `nombre`. Normalizar los nombres (strip, lower, quitar espacios multiples) y luego identificar cuales nombres, tras la normalizacion, resultan ser el mismo. Mostrar los grupos de nombres que son duplicados aproximados.

In [ ]:
# Tu solucion para el Ejercicio 12
# Escribe tu codigo aqui



**Ejercicio 13:** Detectar duplicados basados en el subconjunto de columnas `['nombre', 'departamento']`, conservando solo la primera aparicion (`keep='first'`). Mostrar cuantos registros se eliminan y una muestra de los duplicados encontrados.

In [ ]:
# Tu solucion para el Ejercicio 13
# Escribe tu codigo aqui



**Ejercicio 14:** Detectar outliers en la columna `salario` utilizando el metodo IQR (Rango Intercuartilico). Calcular Q1, Q3, IQR, y los limites inferior y superior. Identificar los outliers y visualizarlos con un boxplot antes y despues de su eliminacion.

In [ ]:
# Tu solucion para el Ejercicio 14
# Escribe tu codigo aqui



**Ejercicio 15:** Detectar outliers en la columna `edad` utilizando el Z-score. Calcular el Z-score de cada valor y marcar como outlier aquellos con |z| > 3. Mostrar los valores detectados y su Z-score correspondiente.

In [ ]:
# Tu solucion para el Ejercicio 15
# Escribe tu codigo aqui



**Ejercicio 16:** Comparar dos estrategias para tratar outliers en `salario`: (a) winsorizar al percentil 1-99 (recortar los extremos) y (b) eliminar los outliers. Visualizar las distribuciones resultantes de ambos metodos lado a lado.

In [ ]:
# Tu solucion para el Ejercicio 16
# Escribe tu codigo aqui



**Ejercicio 17:** Utilizar Isolation Forest de scikit-learn para detectar anomalias multivariantes en las columnas `['edad', 'salario', 'evaluacion']`. Visualizar los puntos anomalos en un scatter plot de edad vs salario, coloreando diferente los puntos normales y los anomalos.

In [ ]:
# Tu solucion para el Ejercicio 17
# Escribe tu codigo aqui



**Ejercicio 18:** Crear una columna con valores monetarios en formato string como "$1,234.56" y "$45,678.90". Limpiar estos strings eliminando el simbolo "$" y las comas, y convertir el resultado a tipo float. Mostrar la columna antes y despues de la conversion.

In [ ]:
# Tu solucion para el Ejercicio 18
# Escribe tu codigo aqui



**Ejercicio 19:** Parsear la columna `fecha_ingreso` que contiene formatos mixtos: algunas fechas estan en formato "YYYY-MM-DD" y otras en "DD/MM/YYYY". Convertir todas a tipo datetime. Mostrar los valores antes y despues de la conversion, incluyendo los que no se pudieron parsear.

In [ ]:
# Tu solucion para el Ejercicio 19
# Escribe tu codigo aqui



**Ejercicio 20:** Validar los datos de `df_dirty` aplicando las siguientes reglas: edad debe estar en el rango [18, 70], salario debe ser mayor que 0, y evaluacion debe estar en el rango [1, 10]. Crear una columna `valido` que indique si la fila cumple todas las reglas. Mostrar el conteo de filas validas e invalidas.

In [ ]:
# Tu solucion para el Ejercicio 20
# Escribe tu codigo aqui



**Ejercicio 21:** Normalizar la columna `nombre` de `df_dirty` aplicando: (1) eliminar espacios al inicio y final (strip), (2) convertir a title case (primera letra de cada palabra en mayuscula), (3) reemplazar multiples espacios consecutivos por uno solo. Mostrar los valores unicos antes y despues.

In [ ]:
# Tu solucion para el Ejercicio 21
# Escribe tu codigo aqui



**Ejercicio 22:** Crear una columna de emails ficticios basados en los nombres de `df_dirty` (ej: "juan.garcia@empresa.com") y extraer el dominio de cada email utilizando expresiones regulares (regex).

In [ ]:
# Tu solucion para el Ejercicio 22
# Escribe tu codigo aqui



**Ejercicio 23:** Estandarizar la columna `departamento` de `df_dirty`: convertir todo a title case y corregir las inconsistencias (Ventas, ventas, VENTAS deben ser todas "Ventas"). Mostrar los valores unicos antes y despues, y la distribucion resultante.

In [ ]:
# Tu solucion para el Ejercicio 23
# Escribe tu codigo aqui



**Ejercicio 24:** Aplicar One-Hot Encoding a la columna `ciudad` (previamente normalizada a title case). Usar `pd.get_dummies()` con `drop_first=True` para evitar la trampa de variables dummy. Mostrar las primeras filas del resultado y explicar por que se elimina una columna.

In [ ]:
# Tu solucion para el Ejercicio 24
# Escribe tu codigo aqui



**Ejercicio 25:** Aplicar Ordinal Encoding a la columna `satisfaccion` con el orden logico: bajo=1, medio=2, alto=3. Primero normalizar los valores (strip, lower) para unificar variantes. Mostrar el mapeo y la distribucion resultante.

In [ ]:
# Tu solucion para el Ejercicio 25
# Escribe tu codigo aqui



**Ejercicio 26:** Aplicar Target Encoding a la columna `departamento` usando la media de `evaluacion` por departamento. Es decir, reemplazar cada valor de departamento por la media de evaluacion de ese departamento. Normalizar departamento antes de aplicar el encoding.

In [ ]:
# Tu solucion para el Ejercicio 26
# Escribe tu codigo aqui



**Ejercicio 27:** Comparar tres metodos de escalado en la columna `salario` (con outliers): MinMaxScaler, StandardScaler y RobustScaler. Mostrar tres histogramas lado a lado con las distribuciones resultantes y discutir cual es mas apropiado en presencia de outliers.

In [ ]:
# Tu solucion para el Ejercicio 27
# Escribe tu codigo aqui



**Ejercicio 28:** Usar `ColumnTransformer` de scikit-learn para aplicar transformaciones diferentes a columnas numericas y categoricas simultaneamente: escalar las numericas (`edad`, `salario`, `evaluacion`) con StandardScaler y aplicar One-Hot Encoding a las categoricas (`departamento`, `ciudad`). Primero preparar los datos eliminando NaN.

In [ ]:
# Tu solucion para el Ejercicio 28
# Escribe tu codigo aqui



**Ejercicio 29:** Construir un pipeline completo con scikit-learn que aplique: para columnas numericas, `SimpleImputer(strategy='median')` seguido de `StandardScaler`; para columnas categoricas, `SimpleImputer(strategy='most_frequent')` seguido de `OneHotEncoder`. Combinar ambos con `ColumnTransformer` y aplicar a `df_dirty`.

In [ ]:
# Tu solucion para el Ejercicio 29
# Escribe tu codigo aqui



**Ejercicio 30:** Limpieza end-to-end. Tomar `df_dirty` original y producir `df_clean` listo para modelo, aplicando TODOS los pasos anteriores en secuencia: (1) eliminar duplicados, (2) normalizar strings, (3) parsear fechas, (4) validar rangos y marcar invalidos, (5) tratar outliers con IQR, (6) imputar nulos, (7) encoding categorico, (8) escalar numericas. Mostrar un resumen comparativo del dataset antes y despues.

In [ ]:
# Tu solucion para el Ejercicio 30
# Escribe tu codigo aqui

